# Random Forest: Feature (Column) Sampling
This notebook implements the exact `sample_features` logic demonstrated in the CampusX video [23:50 - 30:00].


In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier


In [ ]:
# 1. Create a toy dataset
X, y = make_classification(n_samples=100, n_features=5, n_informative=5, n_redundant=0, random_state=42)
df = pd.DataFrame(X, columns=['col1', 'col2', 'col3', 'col4', 'col5'])
df['target'] = y
print('Original Dataset Shape:', df.shape)
display(df.head())


In [ ]:
def sample_features(df, percent):
    '''Samples a percentage of columns from the dataset (without replacement), keeping the target.'''
    cols = list(df.columns)
    cols.remove('target') # Never remove the target column
    n_cols = int(len(cols) * percent)
    
    sampled_cols = random.sample(cols, n_cols)
    sampled_cols.append('target') # Add it back to the end
    
    return df[sampled_cols]


In [ ]:
# 2. Generate Sub-datasets using Column Sampling
df1 = sample_features(df, 0.8) # 80% of columns = 4 columns
df2 = sample_features(df, 0.8)
df3 = sample_features(df, 0.8)

print('Columns in df1:', df1.columns.tolist())
print('Columns in df2:', df2.columns.tolist())
print('Columns in df3:', df3.columns.tolist())


In [ ]:
# 3. Train Decision Trees on sampled columns
tree1 = DecisionTreeClassifier()
tree2 = DecisionTreeClassifier()
tree3 = DecisionTreeClassifier()

tree1.fit(df1.iloc[:, :-1], df1.iloc[:, -1])
tree2.fit(df2.iloc[:, :-1], df2.iloc[:, -1])
tree3.fit(df3.iloc[:, :-1], df3.iloc[:, -1])
print('Decision Trees successfully trained on column-sampled data!')


In [ ]:
# 4. Make Predictions (Majority Count)
# Note: We must pass only the columns that each specific tree was trained on
new_point_df = pd.DataFrame([[0.5, -0.2, 1.1, -0.9, 0.3]], columns=['col1', 'col2', 'col3', 'col4', 'col5'])

pred1 = tree1.predict(new_point_df[df1.columns[:-1]])[0]
pred2 = tree2.predict(new_point_df[df2.columns[:-1]])[0]
pred3 = tree3.predict(new_point_df[df3.columns[:-1]])[0]

print(f'Tree 1 Output: {pred1}')
print(f'Tree 2 Output: {pred2}')
print(f'Tree 3 Output: {pred3}')

# Aggregation
from collections import Counter
final_pred = Counter([pred1, pred2, pred3]).most_common(1)[0][0]
print(f'\nFinal Random Forest Prediction: {final_pred}')
